# Lab 05 — Lakeflow Declarative Pipelines
## 00 — Environment Setup

This notebook performs the **one-time / rerunnable infrastructure setup** for Lab 05.

### Responsibilities
- Validate the target Unity Catalog catalog
- Create the Lab 05 schema if required
- Create the managed reference/test-data volume
- Create the managed streaming/source volume
- Create the source-data directories used by the lab
- Validate that the required directories exist

### This notebook intentionally does **not**
- Create Bronze, Silver, or Gold pipeline datasets
- Run the Lakeflow declarative pipeline
- Create or manage streaming checkpoints
- Create or manage Auto Loader schema locations
- Download Citi Bike source data

Those responsibilities are kept separate so that the project follows a clean separation of concerns:

**Setup → Source preparation → Declarative pipeline → Validation**

Both Lab 05 volumes are Unity Catalog **managed volumes**, so this notebook does not depend on an external location name.


## 1. Runtime parameters

The setup notebook uses four infrastructure parameters:

- `catalog`
- `schema`
- `volume_name` — managed reference/test-data volume
- `streaming_volume_name` — managed volume used for incoming Citi Bike snapshots

Using managed volumes makes the Lab 05 setup portable across the personal and Azure workspaces without requiring workspace-specific external-location names.


In [ ]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "Schema")
dbutils.widgets.text("volume_name", "lab05_lakeflow", "Managed reference volume")
dbutils.widgets.text(
    "streaming_volume_name",
    "lab05_lakeflow_streaming",
    "Managed streaming volume"
)

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
streaming_volume_name = dbutils.widgets.get(
    "streaming_volume_name"
).strip()

print(f"Catalog                 : {catalog}")
print(f"Schema                  : {schema}")
print(f"Reference/test volume   : {volume_name}")
print(f"Streaming/source volume : {streaming_volume_name}")


## 2. Validate identifiers and define storage paths

Before using widget values in SQL object names, validate that they contain only safe identifier characters.

Lab 05 uses two managed Unity Catalog volumes:

```text
/Volumes/<catalog>/<schema>/<volume_name>/
├── reference/
└── test_data/

/Volumes/<catalog>/<schema>/<streaming_volume_name>/
└── landing/
    └── station_status/
```

There is deliberately **no manually managed `checkpoints/` directory** in this design.

Lakeflow owns the streaming state/checkpoints; this setup notebook only creates the source directories required before the producer and pipeline run.


In [ ]:
import re

IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def validate_identifier(name: str, label: str) -> None:
    if not IDENTIFIER_PATTERN.fullmatch(name):
        raise ValueError(
            f"Invalid {label}: {name!r}. "
            "Use letters, numbers, and underscores only."
        )


for value, label in [
    (catalog, "catalog"),
    (schema, "schema"),
    (volume_name, "managed reference volume name"),
    (streaming_volume_name, "managed streaming volume name"),
]:
    validate_identifier(value, label)


volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
streaming_volume_path = (
    f"/Volumes/{catalog}/{schema}/{streaming_volume_name}"
)

status_landing_path = (
    f"{streaming_volume_path}/landing/station_status"
)
reference_path = f"{volume_path}/reference"
test_data_path = f"{volume_path}/test_data"

print(f"Reference volume root : {volume_path}")
print(f"Streaming volume root : {streaming_volume_path}")
print(f"Streaming landing     : {status_landing_path}")
print(f"Reference data        : {reference_path}")
print(f"Test data             : {test_data_path}")


## 3. Validate the catalog and create Lab 05 structural objects

The catalog is treated as shared infrastructure, so this notebook **does not create it**.

Instead it:

1. Verifies that the catalog exists and is accessible.
2. Creates the personal Lab 05 schema if required.
3. Creates the managed reference/test-data volume if required.
4. Creates the managed streaming/source volume if required.

`IF NOT EXISTS` makes the setup safe to rerun before every end-to-end Job execution.


In [ ]:
available_catalogs = {
    row.catalog
    for row in spark.sql("SHOW CATALOGS").collect()
}

if catalog not in available_catalogs:
    raise ValueError(
        f"Catalog '{catalog}' does not exist or is not accessible."
    )

print(f"✅ Catalog exists: {catalog}")

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`"
)

print(f"✅ Schema ready: {catalog}.{schema}")

# Managed volume for reference data and test fixtures.
spark.sql(
    f'''
    CREATE VOLUME IF NOT EXISTS
    `{catalog}`.`{schema}`.`{volume_name}`
    '''
)

print(
    f"✅ Reference/test managed volume ready: "
    f"{catalog}.{schema}.{volume_name}"
)

# Managed volume for incoming Citi Bike streaming snapshots.
spark.sql(
    f'''
    CREATE VOLUME IF NOT EXISTS
    `{catalog}`.`{schema}`.`{streaming_volume_name}`
    '''
)

print(
    f"✅ Streaming/source managed volume ready: "
    f"{catalog}.{schema}.{streaming_volume_name}"
)


## 4. Create source and test-data directories

Only directories required for source/reference/test files are created here.

### Streaming volume — `landing/station_status/`
Receives timestamped Citi Bike `station_status` JSON snapshots. Each producer execution writes **one immutable snapshot file**.

### Reference/test volume — `reference/`
Stores the batch/reference `station_information.json` file.

### Reference/test volume — `test_data/`
Stores controlled invalid or test fixtures used to prove expectation behavior.

Bronze, Silver, and Gold tables are **not** created here. They are declared inside `pipeline/bronze.py`, `pipeline/silver.py`, and `pipeline/gold.py`.


In [ ]:
directories = [
    status_landing_path,
    reference_path,
    test_data_path,
]

for path in directories:
    dbutils.fs.mkdirs(path)
    print(f"✅ Ready: {path}")


## 5. Inspect the created directory structure

This step provides quick visual evidence that both managed volumes and their required directories are ready before source preparation begins.


In [ ]:
display(dbutils.fs.ls(volume_path))


In [ ]:
display(dbutils.fs.ls(f"{streaming_volume_path}/landing"))


## 6. Final setup validation

Instead of assuming that directory creation succeeded, perform explicit validation.

This follows the same principle used in previous labs: setup should finish with a clear **PASS/FAIL** result rather than leaving infrastructure problems to surface later inside the production pipeline.


In [ ]:
required_directories = {
    "station_status_landing": status_landing_path,
    "reference_data": reference_path,
    "test_data": test_data_path,
}

validation_results = []

for name, path in required_directories.items():
    try:
        dbutils.fs.ls(path)
        validation_results.append((name, path, "PASS"))
    except Exception as exc:
        validation_results.append(
            (name, path, f"FAIL: {exc}")
        )

validation_df = spark.createDataFrame(
    validation_results,
    ["check", "path", "result"]
)

display(validation_df)


In [ ]:
failed_checks = [
    row
    for row in validation_results
    if not row[2].startswith("PASS")
]

if failed_checks:
    raise AssertionError(
        f"Lab 05 setup failed: {failed_checks}"
    )

print("✅ LAB 05 SETUP PASSED")
print()
print(f"Reference/test volume : {volume_path}")
print(f"Streaming/source volume: {streaming_volume_path}")
print(f"Streaming landing      : {status_landing_path}")
print(f"Reference data         : {reference_path}")
print(f"Test data              : {test_data_path}")


## Expected result

A successful run should end with:

```text
✅ LAB 05 SETUP PASSED

Reference/test volume : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow
Streaming/source volume: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming
Streaming landing      : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status
Reference data         : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference
Test data              : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/test_data
```

### Ownership boundary after this notebook

| Component | Owner |
|---|---|
| Catalog validation | `lab05_00_setup` |
| Schema | `lab05_00_setup` |
| Managed reference/test volume | `lab05_00_setup` |
| Managed streaming/source volume | `lab05_00_setup` |
| Landing/reference/test directories | `lab05_00_setup` |
| Citi Bike source download | `lab05_01_source_preparation` |
| Streaming snapshot producer | `tools/citibike_status_producer.py` |
| Bronze datasets | Lakeflow pipeline |
| Silver datasets + expectations | Lakeflow pipeline |
| Gold dataset | Lakeflow pipeline |
| Streaming state/checkpoints | Lakeflow |
| Final validation | `lab05_02_validation` |

### Next step

Continue with `notebooks/lab05_01_source_preparation.ipynb`. That notebook downloads `station_information.json`, fetches initial `station_status` snapshots, and profiles both sources before the declarative pipeline runs.
